In [ ]:
# pip install pystac-client pystac requests


In [1]:
import re
import time
from urllib.parse import urljoin

import requests

COLLECTION_URL = "https://s3.eu-central-1.wasabisys.com/stac/openlandmap/lst_mod11a2.daytime/collection.json"
TARGET_YEAR = 2021

session = requests.Session()


def fetch_json(url, attempts=5, timeout=30):
    last_error = None
    for i in range(attempts):
        try:
            response = session.get(url, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except Exception as err:
            last_error = err
            time.sleep(0.5 * (i + 1))
    raise RuntimeError(f"Failed to fetch JSON after {attempts} attempts: {url}") from last_error


collection = fetch_json(COLLECTION_URL)
item_links = [
    urljoin(COLLECTION_URL, link["href"])
    for link in collection["links"]
    if link.get("rel") == "item"
]

items_by_year = {}
failed_items = []

for href in item_links:
    try:
        item = fetch_json(href)
        match = re.search(r"(19|20)\d{2}", item["id"])
        if match:
            year = int(match.group(0))
            items_by_year[year] = item
    except Exception:
        failed_items.append(href)

years = sorted(items_by_year)
print(f"Loaded {len(years)} yearly items: {years[0]} to {years[-1]}")

if failed_items:
    print(f"Skipped {len(failed_items)} unreachable item(s).")

if TARGET_YEAR not in items_by_year:
    print(f"Year {TARGET_YEAR} is not currently available in loaded items.")
else:
    item = items_by_year[TARGET_YEAR]
    print(f"\nFound item for {TARGET_YEAR}: {item['id']}")

    print("\nAvailable Assets:")
    for asset_key, asset_value in item["assets"].items():
        print(f"- {asset_key}: {asset_value['href']}")

    preferred = "lst_mod11a2.daytime_p50_1km_s"
    if preferred in item["assets"]:
        print(f"\nDirect Download Link ({preferred}): {item['assets'][preferred]['href']}")
    else:
        first_key = next(iter(item["assets"].keys()))
        print(f"\nDirect Download Link ({first_key}): {item['assets'][first_key]['href']}")


Loaded 22 yearly items: 2000 to 2021

Found item for 2021: lst_mod11a2.daytime_20211201_20211231

Available Assets:
- lst_mod11a2.daytime_p50_1km_s: https://s3.openlandmap.org/arco/lst_mod11a2.daytime_p50_1km_s_20211201_20211231_go_epsg.4326_v1.2.tif
- lst_mod11a2.daytime_p05_1km_s: https://s3.openlandmap.org/arco/lst_mod11a2.daytime_p05_1km_s_20211201_20211231_go_epsg.4326_v1.2.tif
- lst_mod11a2.daytime_p95_1km_s: https://s3.openlandmap.org/arco/lst_mod11a2.daytime_p95_1km_s_20211201_20211231_go_epsg.4326_v1.2.tif
- lst_mod11a2.daytime_sd_1km_s: https://s3.openlandmap.org/arco/lst_mod11a2.daytime_sd_1km_s_20211201_20211231_go_epsg.4326_v1.2.tif
- sld: https://s3.openlandmap.org/arco/lst_mod11a2_m.sld
- qml: https://s3.openlandmap.org/arco/lst_mod11a2_m.qml
- thumbnail: https://s3.eu-central-1.wasabisys.com/stac/openlandmap/lst_mod11a2.daytime/lst_mod11a2.daytime_20211201_20211231/lst_mod11a2.daytime_p50_1km_s_20211201_20211231_go_epsg.4326_v1.2.png
- lst_mod11a2.daytime_p05_1km_s_prev

In [ ]:
import json
from html import escape
from pathlib import Path
from urllib.parse import quote_plus
from xml.etree import ElementTree as ET

import folium
from folium.plugins import SideBySideLayers
import requests
from IPython.display import IFrame, display

YEAR_LEFT = 2000
YEAR_RIGHT = 2021

def get_tile_url(year, items_dict):
    if year not in items_dict:
        return None, None
    item = items_dict[year]
    preferred = "lst_mod11a2.daytime_p50_1km_s"
    asset_href = item["assets"].get(preferred, {}).get("href")
    if not asset_href:
        asset_href = next(iter(item["assets"].values()))["href"]
    
    style_href = item["assets"].get("sld", {}).get("href")
    colormap_dict = {}
    legend_entries = []
    
    if style_href:
        style_xml = requests.get(style_href, timeout=60).text
        style_root = ET.fromstring(style_xml)
        namespace = {"sld": "http://www.opengis.net/sld"}
        
        # Check colormap type
        cmap_elem = style_root.find(".//sld:ColorMap", namespace)
        is_ramp = cmap_elem is not None and cmap_elem.attrib.get("type") == "ramp"
        
        entries = []
        for entry in style_root.findall(".//sld:ColorMapEntry", namespace):
            quantity = entry.attrib.get("quantity")
            label = entry.attrib.get("label") or quantity
            color = entry.attrib.get("color")
            if color and quantity:
                entries.append((float(quantity), color, label))

        if is_ramp and len(entries) > 1:
            entries.sort()
            for i in range(len(entries) - 1):
                v1, c1, l1 = entries[i]
                v2, c2, l2 = entries[i+1]
                legend_entries.append((l1, c1))
                
                # Interpolate to create a dense colormap for continuous data
                # TiTiler limit is around 4000 entries; 20-50 steps per segment is safe.
                step = max(1, (v2 - v1) / 20)
                for j in range(21):
                    v = v1 + j * step
                    if v <= v2:
                        colormap_dict[str(int(v))] = c1
            legend_entries.append((entries[-1][2], entries[-1][1]))
            colormap_dict[str(int(entries[-1][0]))] = entries[-1][1]
        else:
            for v, c, l in entries:
                colormap_dict[str(int(v))] = c
                legend_entries.append((l, c))
    
    base_url = "https://titiler.xyz/cog/tiles/WebMercatorQuad/{z}/{x}/{y}.png?url=" + quote_plus(asset_href)
    if colormap_dict:
        base_url += f"&colormap={quote_plus(json.dumps(colormap_dict))}"
    
    return base_url, legend_entries

url_left, legend_left = get_tile_url(YEAR_LEFT, items_by_year)
url_right, legend_right = get_tile_url(YEAR_RIGHT, items_by_year)

if not url_left or not url_right:
    raise ValueError(f"Missing data for {YEAR_LEFT} or {YEAR_RIGHT}")

m = folium.Map(location=[41.2974, 2.0833], zoom_start=12, tiles="OpenStreetMap", control_scale=True)

# Add compatibility shim
shim = """
<script>
if (typeof L !== 'undefined') {
    if (typeof L.Mixin === 'undefined') L.Mixin = {};
    if (L.Mixin && typeof L.Mixin.Events === 'undefined' && L.Evented) {
        L.Mixin.Events = L.Evented.prototype;
    }
}
</script>
"""
m.get_root().header.add_child(folium.Element(shim))

layer_left = folium.TileLayer(
    tiles=url_left,
    name=f"LST_MOD11A2 DAYTIME {YEAR_LEFT}",
    attr="OpenLandMap",
    overlay=True,
    max_zoom=24,
).add_to(m)

layer_right = folium.TileLayer(
    tiles=url_right,
    name=f"LST_MOD11A2 DAYTIME {YEAR_RIGHT}",
    attr="OpenLandMap",
    overlay=True,
    max_zoom=24,
).add_to(m)

SideBySideLayers(layer_left, layer_right).add_to(m)

legend_items = "".join(
    f"<div style='display:flex; align-items:center; gap:8px; margin:4px 0;'>"
    f"<span style='width:14px; height:14px; background:{color}; border:1px solid #666; display:inline-block;'></span>"
    f"<span style='font-size:12px; line-height:1.2;'>{escape(label)}</span></div>"
    for label, color in legend_left
)

left_id = layer_left.get_name()
right_id = layer_right.get_name()

# Ensure layers are on window for the opacity script
fix_names_js = f"window.{left_id} = {left_id}; window.{right_id} = {right_id};"
m.get_root().html.add_child(folium.Element(f"<script>{fix_names_js}</script>"))

top_labels_html = f'''
<div style="position: fixed; top: 20px; left: 60px; z-index: 9999; background: rgba(255,255,255,0.9); padding: 8px 16px; border: 2px solid #333; border-radius: 6px; font-weight: 800; font-family: sans-serif; font-size: 16px; box-shadow: 0 2px 5px rgba(0,0,0,0.2);">{YEAR_LEFT}</div>
<div style="position: fixed; top: 20px; right: 20px; z-index: 9999; background: rgba(255,255,255,0.9); padding: 8px 16px; border: 2px solid #333; border-radius: 6px; font-weight: 800; font-family: sans-serif; font-size: 16px; box-shadow: 0 2px 5px rgba(0,0,0,0.2);">{YEAR_RIGHT}</div>
'''

legend_html = f'''
<div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999; background: rgba(255,255,255,0.95); padding: 12px 14px; border: 1px solid #999; border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.15); font-family: sans-serif; max-width: 320px;">
    <div style="font-weight: 700; margin-bottom: 8px;">LST_MOD11A2 DAYTIME Legend ({YEAR_LEFT} vs {YEAR_RIGHT})</div>
    
    <div style="margin-bottom: 15px; padding-bottom: 10px; border-bottom: 1px solid #eee;">
        <div style="font-size: 11px; font-weight: bold; margin-bottom: 5px; color: #333;">Render Opacity: <span id="opacity-val">1.0</span></div>
        <input id="opacity-slider" type="range" min="0" max="1" step="0.05" value="1" style="width: 100%; cursor: pointer;">
    </div>

    <div style="max-height: 250px; overflow-y: auto; padding-right: 4px;">
        {legend_items}
    </div>
</div>
'''

opacity_script = f'''
<script>
    document.addEventListener("DOMContentLoaded", function() {{
        var slider = document.getElementById("opacity-slider");
        var label = document.getElementById("opacity-val");
        var renderLayers = ["{left_id}", "{right_id}"];
        
        function updateOpacity(val) {{
            label.innerText = parseFloat(val).toFixed(2);
            renderLayers.forEach(function(id) {{
                if (window[id] && typeof window[id].setOpacity === "function") {{
                    window[id].setOpacity(val);
                }}
            }});
        }}
        
        if (slider) {{
            slider.addEventListener("input", function() {{
                updateOpacity(this.value);
            }});
        }}
    }});
</script>
'''

m.get_root().html.add_child(folium.Element(top_labels_html + legend_html + opacity_script))
map_path = Path("lst_mod11a2_daytime_map.html").resolve()
m.save(str(map_path))
display(IFrame(src=map_path.as_uri(), width="100%", height=700))
